In [ ]:
%load_ext autoreload
%autoreload 2


import numpy as np
import matplotlib.pyplot as plt

#import uproot
import pandas as pd

In [ ]:
import xgboost as xgb
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.datasets import make_classification

In [ ]:
from matplotlib.colors import LogNorm

In [ ]:
plt.rcParams.update({'font.size': 14})

In [ ]:
# Open dataframe from CSV

In [ ]:
df = pd.read_csv('pi0_events.csv')

In [ ]:
print(df.keys())

### Pandas Lambda vs. direct calculation

In [ ]:
import time

start = time.time()
df['reco_etot'] = df['reco_g1_e'] + df['reco_g2_e']
end = time.time()
print('calculation took %g sec'%(end-start))

In [ ]:
def etot(x):
    return x['reco_g1_e'] + x['reco_g2_e']

In [ ]:
start = time.time()
df['reco_etot_lambda'] = df.apply(lambda x: etot(x), axis=1)
end = time.time()
print('calculation took %g sec'%(end-start))

### True variables

In [ ]:
def true_g1_x_diff(x):
    return x['true_g1_x'] - x['true_nu_x']

def true_g1_y_diff(x):
    return x['true_g1_y'] - x['true_nu_y']

def true_g1_z_diff(x):
    return x['true_g1_z'] - x['true_nu_z']

def true_g2_x_diff(x):
    return x['true_g2_x'] - x['true_nu_x']

def true_g2_y_diff(x):
    return x['true_g2_y'] - x['true_nu_y']

def true_g2_z_diff(x):
    return x['true_g2_z'] - x['true_nu_z']

In [ ]:
df['true_g1_x_diff'] = df.apply(lambda x: true_g1_x_diff(x), axis=1)
df['true_g1_y_diff'] = df.apply(lambda x: true_g1_y_diff(x), axis=1)
df['true_g1_z_diff'] = df.apply(lambda x: true_g1_z_diff(x), axis=1)

df['true_g2_x_diff'] = df.apply(lambda x: true_g2_x_diff(x), axis=1)
df['true_g2_y_diff'] = df.apply(lambda x: true_g2_y_diff(x), axis=1)
df['true_g2_z_diff'] = df.apply(lambda x: true_g2_z_diff(x), axis=1)

In [ ]:
def true_g1_dist(x):
    return (x['true_g1_x_diff']**2 + x['true_g1_y_diff']**2 + x['true_g1_z_diff']**2)**(0.5)

def true_g2_dist(x):
    return (x['true_g2_x_diff']**2 + x['true_g2_y_diff']**2 + x['true_g2_z_diff']**2)**(0.5)

In [ ]:
df['true_g1_dist'] = df.apply(lambda x: true_g1_dist(x), axis=1)
df['true_g2_dist'] = df.apply(lambda x: true_g2_dist(x), axis=1)

In [ ]:
df['true_g1_x_dir'] = df['true_g1_x_diff'] / df['true_g1_dist']
df['true_g1_y_dir'] = df['true_g1_y_diff'] / df['true_g1_dist']
df['true_g1_z_dir'] = df['true_g1_z_diff'] / df['true_g1_dist']

df['true_g2_x_dir'] = df['true_g2_x_diff'] / df['true_g2_dist']
df['true_g2_y_dir'] = df['true_g2_y_diff'] / df['true_g2_dist']
df['true_g2_z_dir'] = df['true_g2_z_diff'] / df['true_g2_dist']

In [ ]:
df['true_gammadot'] = (df['true_g1_x_dir']*df['true_g2_x_dir']) + (df['true_g1_y_dir']*df['true_g2_y_dir']) + (df['true_g1_z_dir']*df['true_g2_z_dir'])

In [ ]:
def true_pi0_mass(x):
    return (2*x['true_g1_e']*x['true_g2_e']*(1-x['true_gammadot']))**(0.5)

In [ ]:
df['true_pi0_mass'] = df.apply(lambda x: true_pi0_mass(x), axis=1)

### Reconstructed Quantities

In [ ]:
df['reco_gammadot'] = (df['reco_g1_px']*df['reco_g2_px'])+(df['reco_g1_py']*df['reco_g2_py'])+(df['reco_g1_pz']*df['reco_g2_pz'])

In [ ]:
def reco_pi0_mass(x):
    return (2*x['reco_g1_e']*x['reco_g2_e']*(1-x['reco_gammadot']))**(0.5)

In [ ]:
df['reco_pi0_mass'] = df.apply(lambda x: reco_pi0_mass(x), axis=1)

### True Pi0 and non-Pi0 Samples

In [ ]:
dfpi0    = df.query('true_pi0_e > 0')
dfnonpi0 = df.query('true_pi0_e <= 0')

In [ ]:
fig = plt.figure(figsize=(6,6))
plt.hist(dfpi0['true_pi0_mass'].values,bins=np.linspace(0,200,51),histtype='step',lw=2,color='k')
plt.axvline(135.,color='k',linestyle='--',lw=2)
plt.xlabel(r'$\pi^0$ Mass [MeV]')
plt.ylabel('counts')
plt.show()

In [ ]:
fig = plt.figure(figsize=(6,6))
plt.hist(dfpi0['reco_pi0_mass'].values,bins=np.linspace(20,500,49),histtype='step',lw=2,color='b',label=r'true $\pi^0$')
plt.hist(dfnonpi0['reco_pi0_mass'].values,bins=np.linspace(20,500,49),histtype='step',lw=2,color='r',label=r'background')
plt.xlabel(r'Reconstructed $\pi^0$ Mass [MeV]')
plt.ylabel('counts')
plt.axvline(135.,color='k',linestyle='--',lw=2)
plt.legend(loc=1)
plt.show()

### Calibrate Energy

In [ ]:
def reco_pi0_mass_calib(x):
    return (2*x['reco_g1_e_calib']*x['reco_g2_e_calib']*(1-x['reco_gammadot']))**(0.5)

In [ ]:
df['reco_g1_e_calib'] = df['reco_g1_e']/0.76
df['reco_g2_e_calib'] = df['reco_g2_e']/0.76

df['reco_pi0_mass_calib'] = df.apply(lambda x: reco_pi0_mass_calib(x), axis=1)

In [ ]:
dfreco = df.query('reco_g1_e > 0')

In [ ]:
dfpi0reco = dfreco.query('true_pi0_e > 0')
dfnonpi0reco = dfreco.query('true_pi0_e <= 0')

In [ ]:
fig = plt.figure(figsize=(6,6))
VAR = 'reco_pi0_mass_calib'
BINS = np.linspace(20,500,49)
plt.hist(dfpi0reco[VAR].values,bins=BINS,histtype='step',lw=2,color='b',label=r'true $\pi^0$')
plt.hist(dfnonpi0reco[VAR].values,bins=BINS,histtype='step',lw=2,color='r',label=r'background')
plt.xlabel(r'Reconstructed $\pi^0$ Mass [MeV]')
plt.ylabel('counts')
plt.axvline(135.,color='k',linestyle='--',lw=2)
plt.legend(loc=1)
plt.show()

In [ ]:
fig = plt.figure(figsize=(6,6))
BINS = np.linspace(-1,1,51)
VAR = 'reco_g1_px'
plt.hist(dfpi0reco[VAR].values   ,bins=BINS,histtype='step',lw=2,color='b',label=r'true $\pi^0$')
plt.hist(dfnonpi0reco[VAR].values,bins=BINS,histtype='step',lw=2,color='r',label=r'background')
plt.xlabel(VAR)
plt.ylabel('counts')
plt.legend(loc=1)
plt.show()

In [ ]:
fig = plt.figure(figsize=(6,6))
BINS = np.linspace(-1,1,51)
VAR = 'reco_g1_py'
plt.hist(dfpi0reco[VAR].values   ,bins=BINS,histtype='step',lw=2,color='b',label=r'true $\pi^0$')
plt.hist(dfnonpi0reco[VAR].values,bins=BINS,histtype='step',lw=2,color='r',label=r'background')
plt.xlabel(VAR)
plt.ylabel('counts')
plt.legend(loc=1)
plt.show()

In [ ]:
fig = plt.figure(figsize=(6,6))
BINS = np.linspace(-1,1,51)
VAR = 'reco_g1_pz'
plt.hist(dfpi0reco[VAR].values   ,bins=BINS,histtype='step',lw=2,color='b',label=r'true $\pi^0$')
plt.hist(dfnonpi0reco[VAR].values,bins=BINS,histtype='step',lw=2,color='r',label=r'background')
plt.xlabel(VAR)
plt.ylabel('counts')
plt.legend(loc=1)
plt.show()

In [ ]:
fig = plt.figure(figsize=(6,6))
BINS = np.linspace(0,500,51)
VAR = 'reco_g1_e_calib'
plt.hist(dfpi0reco[VAR].values   ,bins=BINS,histtype='step',lw=2,color='b',label=r'true $\pi^0$')
plt.hist(dfnonpi0reco[VAR].values,bins=BINS,histtype='step',lw=2,color='r',label=r'background')
plt.xlabel(VAR)
plt.ylabel('counts')
plt.legend(loc=1)
plt.show()

In [ ]:
fig = plt.figure(figsize=(6,6))
BINS = np.linspace(0,500,51)
VAR = 'reco_g2_e_calib'
plt.hist(dfpi0reco[VAR].values   ,bins=BINS,histtype='step',lw=2,color='b',label=r'true $\pi^0$')
plt.hist(dfnonpi0reco[VAR].values,bins=BINS,histtype='step',lw=2,color='r',label=r'background')
plt.xlabel(VAR)
plt.ylabel('counts')
plt.legend(loc=1)
plt.show()

In [ ]:
fig = plt.figure(figsize=(6,6))
BINS = np.linspace(0,10,51)
VAR = 'reco_dedx1_e'
plt.hist(dfpi0reco[VAR].values   ,bins=BINS,histtype='step',lw=2,color='b',label=r'true $\pi^0$')
plt.hist(dfnonpi0reco[VAR].values,bins=BINS,histtype='step',lw=2,color='r',label=r'background')
plt.xlabel(VAR)
plt.ylabel('counts')
plt.legend(loc=1)
plt.show()

In [ ]:
fig = plt.figure(figsize=(6,6))
BINS = np.linspace(0,10,51)
VAR = 'reco_dedx2_e'
plt.hist(dfpi0reco[VAR].values   ,bins=BINS,histtype='step',lw=2,color='b',label=r'true $\pi^0$')
plt.hist(dfnonpi0reco[VAR].values,bins=BINS,histtype='step',lw=2,color='r',label=r'background')
plt.xlabel(VAR)
plt.ylabel('counts')
plt.legend(loc=1)
plt.show()

In [ ]:
fig = plt.figure(figsize=(6,6))
BINS = np.linspace(0,100,51)
VAR = 'pi0_radlen1'
plt.hist(dfpi0reco[VAR].values   ,bins=BINS,histtype='step',lw=2,color='b',label=r'true $\pi^0$')
plt.hist(dfnonpi0reco[VAR].values,bins=BINS,histtype='step',lw=2,color='r',label=r'background')
plt.xlabel(VAR)
plt.ylabel('counts')
plt.yscale('log')
plt.legend(loc=1)
plt.show()

In [ ]:
fig = plt.figure(figsize=(6,6))
BINS = np.linspace(0,100,51)
VAR = 'pi0_radlen2'
plt.hist(dfpi0reco[VAR].values   ,bins=BINS,histtype='step',lw=2,color='b',label=r'true $\pi^0$')
plt.hist(dfnonpi0reco[VAR].values,bins=BINS,histtype='step',lw=2,color='r',label=r'background')
plt.xlabel(VAR)
plt.ylabel('counts')
plt.yscale('log')
plt.legend(loc=1)
plt.show()

### BDT Training

In [ ]:
import xgboost as xgb
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.datasets import make_classification

In [ ]:
df['pi0'] = df['true_pi0_e'].apply(lambda x: 1 if x > 0 else 0)

In [ ]:
# all features including pi0 mass
features = ['reco_g2_e_calib','reco_g1_e_calib','reco_g1_pz','reco_g1_py','reco_g1_px',\
            'reco_g2_pz','reco_g2_py','reco_g2_px',\
           'pi0_radlen2','pi0_radlen1','reco_dedx2_e','reco_dedx1_e','reco_pi0_mass_calib']

# all non-pi0 mass features
features = ['reco_g2_e_calib','reco_g1_e_calib','reco_g1_pz','reco_g1_py','reco_g1_px',\
            'reco_g2_pz','reco_g2_py','reco_g2_px',\
           'pi0_radlen2','pi0_radlen1','reco_dedx2_e','reco_dedx1_e']

# non-kinematics features
#features = ['pi0_radlen2','pi0_radlen1','reco_dedx2_e','reco_dedx1_e']

# kinematics only
#features = ['reco_g2_e_calib','reco_g1_e_calib','reco_g1_pz','reco_g1_py','reco_g1_px',\
#            'reco_g2_pz','reco_g2_py','reco_g2_px']

X = df[features]

target = 'pi0'
y = df[target]

In [ ]:
# Then split and train exactly as before
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=20,       # number of trees [100]
    max_depth=7,            # shallow trees work well for few features [3]
    learning_rate=0.1,      # aka eta
    subsample=0.8,          # row subsampling per tree
    colsample_bytree=0.8,   # feature subsampling per tree
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)

In [ ]:
model.fit(X_train, y_train,eval_set=[(X_test, y_test)],verbose=False)

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}")

xgb.plot_importance(model, max_num_features=10)
plt.tight_layout()
plt.show()

In [ ]:
#results = model.evals_result()
#train_loss = results["validation_0"]["logloss"]
#test_loss  = results["validation_1"]["logloss"]

In [ ]:
# Add a column flagging train vs test
df["split"] = "train"
df.loc[X_test.index, "split"] = "test"

# Then score everything
df["bdt_score"] = model.predict_proba(df[features])[:, 1]

In [ ]:
print(df['bdt_score'])

In [ ]:
fig = plt.figure(figsize=(6,6))
BINS2D = (np.linspace(10,300,51),np.linspace(0,1,51))
plt.hist2d(df['reco_pi0_mass'],df['bdt_score'],bins=BINS2D)
plt.xlabel(r'$\pi^0$ Mass [MeV]')
plt.ylabel('BDT score')
plt.show()

In [ ]:
dfpi0    = df.query('pi0==1')
dfnonpi0 = df.query('pi0==0')

In [ ]:
fig = plt.figure(figsize=(6,6))
BINS = np.linspace(0,1,101)
plt.hist(dfpi0['bdt_score']   ,bins=BINS,histtype='step',lw=2,color='b',label=r'$\pi^0$')
plt.hist(dfnonpi0['bdt_score'],bins=BINS,histtype='step',lw=2,color='r',label=r'background')
plt.xlabel('BDT Score')
plt.yscale('log')
plt.legend(loc=1)
plt.show()

In [ ]:
dfsel = df.query('bdt_score > 0.7')

In [ ]:
fig = plt.figure(figsize=(6,6))
BINS = np.linspace(20,500,49)
plt.hist(df['reco_pi0_mass'].values   ,bins=BINS,histtype='step',lw=2,color='k',label=r'all')
plt.hist(dfpi0['reco_pi0_mass'].values,bins=BINS,histtype='step',lw=2,color='b',label=r'signal')
plt.hist(dfsel['reco_pi0_mass'].values,bins=BINS,histtype='step',lw=2,color='r',label=r'selected')
plt.xlabel(r'Reconstructed $\pi^0$ Mass [MeV]')
plt.ylabel('counts')
plt.axvline(135.,color='k',linestyle='--',lw=2)
plt.legend(loc=1)
plt.show()